In [ ]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


In [ ]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

In [ ]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_14/class2/class2'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


# Lab 2: AI System Testing Framework (Student Version)

## Overview
Build a complete AI system testing framework covering all three layers from Class 2.

### What You'll Build:
- **Phase A**: CNN API with validation test suite (Interface Layer)
- **Phase B**: Drift detection with KL divergence (Data Layer)
- **Phase C**: Model comparison with traffic routing (Deployment Layer)

### Deliverables:
1. Working CNN API with `/predict`, `/health`, `/info`
2. API test suite (8+ test cases)
3. Drift detection script with KL divergence
4. Model comparison report

### Time Allocation:
- **Phase A**: 30 min
- **Phase B**: 30 min
- **Phase C**: 25 min
- **Analysis**: 15 min

---

## Submission Checklist
- [ ] Phase A: API endpoints working, 8+ tests passing
- [ ] Phase B: Drift detector detecting 2+ drift scenarios
- [ ] Phase C: Multi-model comparison with traffic routing
- [ ] All TODO items completed
- [ ] All 5 analysis questions answered
- [ ] Code runs without errors

In [ ]:
%%jshell

runPy("""
import os
import io
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageEnhance
from datetime import datetime
from collections import defaultdict
from typing import Dict, List

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

import matplotlib.pyplot as plt
from scipy.stats import entropy

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['animal', 'name_board', 'pedestrian', 'pothole', 'road_sign', 'speed_breaker', 'vehicle']
NUM_CLASSES = len(CLASS_NAMES)
DATASET_PATH = str(REPO_ROOT / 'module_4/dataset/dataset')

print(f"Device: {DEVICE}")
print(f"Classes ({NUM_CLASSES}): {', '.join(CLASS_NAMES)}")
""");


---
# Phase A: Interface Module — CNN API + Validation Tests

Build a single-model FastAPI with `/predict`, `/health`, `/info` and write 8+ test cases.

Reference: Part 1 notebook

In [ ]:
%%jshell

runPy("""
# TODO: Create the CNN Model class
# Hints:
# - Use resnet18(weights=None) as base
# - Replace final fc layer: nn.Linear(in_features, NUM_CLASSES)
# - forward() calls self.resnet(x)

class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # TODO: Initialize ResNet-18 and modify final layer
        pass

    def forward(self, x):
        # TODO: Forward pass
        pass

print("TODO: CNNModel needs implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Create model instance and transformation pipeline
# Hints:
# - model = CNNModel(NUM_CLASSES).to(DEVICE); model.eval()
# - transform = transforms.Compose([Resize(128,128), ToTensor(), Normalize(...)])

model = None   # TODO
transform = None  # TODO

if model and transform:
    print("Model and transform ready")
else:
    print("TODO: Initialize model and transform")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Define response schemas and create FastAPI app
# Hints:
# - PredictionResponse: prediction (str), confidence (float), model_version (str), latency_ms (float)
# - HealthResponse: status, model_version, device, timestamp

class PredictionResponse(BaseModel):
    # TODO: Define fields
    pass

app = None  # TODO: FastAPI(title="ADAS Classifier API", version="1.0")

print("TODO: Response schemas and app need definition")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Implement /health, /info, and /predict endpoints
# Hints for /predict:
# - Validate file extension (.jpg, .jpeg, .png, .gif, .bmp) -> 400 if invalid
# - Read file, check not empty -> 400
# - Open with PIL, check dimensions >= 32x32 -> 400
# - Transform, run model, return prediction + confidence
# - Catch unexpected errors -> 500
# - Missing file param -> 422 (automatic from FastAPI)

if app is not None:
    # TODO: @app.get("/health")
    # TODO: @app.get("/info")
    # TODO: @app.post("/predict")
    pass

print("TODO: Endpoints need implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Write 8+ test cases
# Create TestClient and helper function first
# Hint: create_test_image(width, height, color, fmt) -> BytesIO

def create_test_image(width=100, height=100, color=(255,0,0), fmt='PNG'):
    # TODO: Create image, save to BytesIO, return
    pass

test_results = []

if app is not None:
    client = TestClient(app)

    # TEST 1: Health check -> 200
    # TODO

    # TEST 2: Model info -> 200, has num_classes=7
    # TODO

    # TEST 3: Valid PNG prediction -> 200, has prediction field
    # TODO

    # TEST 4: Valid JPEG -> 200
    # TODO

    # TEST 5: Invalid file type (.txt) -> 400
    # TODO

    # TEST 6: Corrupt image -> 400
    # TODO

    # TEST 7: Tiny image (16x16) -> 400
    # TODO

    # TEST 8: Missing file -> 422
    # TODO

    passed = sum(1 for _, _, _, p in test_results if p)
    print(f"\nResults: {passed}/{len(test_results)} tests passed")
else:
    print("TODO: App not initialized")
""");


---
# Phase B: Data Monitoring Module — Drift Detection

Create shifted datasets, compute KL divergence, and monitor accuracy under drift.

Reference: Part 2 notebook

In [ ]:
%%jshell

runPy("""
# TODO: Create BrightnessShiftedDataset
# Hints:
# - __init__: store original_dataset, brightness_factor, transform_base
# - __getitem__: load image from imgs[idx], apply ImageEnhance.Brightness, then transform
# - __len__: return len(original_dataset)

class BrightnessShiftedDataset(torch.utils.data.Dataset):
    def __init__(self, original_dataset, brightness_factor=0.1):
        # TODO
        pass

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, idx):
        # TODO
        pass

print("TODO: BrightnessShiftedDataset needs implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Create ContrastShiftedDataset (similar pattern)

class ContrastShiftedDataset(torch.utils.data.Dataset):
    def __init__(self, original_dataset, contrast_factor=0.2):
        # TODO
        pass

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, idx):
        # TODO
        pass

print("TODO: ContrastShiftedDataset needs implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Load datasets and evaluate model accuracy
# Hints:
# - Load test_dataset = ImageFolder(TEST_PATH, transform=transform_base)
# - Create brightness_shifted and contrast_shifted datasets
# - Write evaluate_dataset(model, loader, device) function
# - Evaluate on all three datasets

transform_base = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def evaluate_dataset(model, loader, device):
    # TODO: Evaluate and return accuracy
    pass

accuracies = {}  # TODO: Populate with 'Original', 'Brightness', 'Contrast'

print("TODO: Dataset evaluation needs implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Implement DriftDetector class
# Hints:
# - __init__: baseline_accuracy, accuracy_threshold=5.0, kl_threshold=0.5, alerts=[]
# - check_accuracy_drift(current, name): returns alert dict with 'triggered' bool
# - check_distribution_drift(kl_score, name): returns alert dict with 'triggered' bool
# - generate_report(): returns dict with total_checks, alerts_triggered, recommendation

class DriftDetector:
    def __init__(self, baseline_accuracy, accuracy_threshold=5.0, kl_threshold=0.5):
        # TODO
        pass

    def check_accuracy_drift(self, current_accuracy, dataset_name):
        # TODO
        pass

    def check_distribution_drift(self, kl_score, dataset_name):
        # TODO
        pass

    def generate_report(self):
        # TODO
        pass

print("TODO: DriftDetector needs implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Run drift detection and display results
# Hints:
# - Create detector with baseline from accuracies['Original']
# - Check accuracy drift for brightness and contrast
# - Generate report and print recommendation

print("TODO: Run drift detection")
""");


---
# Phase C: Deployment Module — Model Comparison + Traffic Routing

Serve two model versions, simulate A/B traffic, and compare performance.

Reference: Part 3 notebook

In [ ]:
%%jshell

runPy("""
# TODO: Create BaselineModel and DropoutModel
# Hints:
# - BaselineModel: same as CNNModel (resnet18 + fc)
# - DropoutModel: resnet18 + Sequential(Dropout(0.5), Linear(512, 7))

class BaselineModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # TODO
        pass

    def forward(self, x):
        # TODO
        pass

class DropoutModel(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        # TODO
        pass

    def forward(self, x):
        # TODO
        pass

print("TODO: Model classes need implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Create multi-model FastAPI with canary routing
# Hints:
# - CANARY_PERCENTAGE = 30 (30% to v2)
# - /predict: random.random() < CANARY_PERCENTAGE/100 decides which model
# - /predict-both: run both models, return ComparisonResponse with agreement
# - /metrics: return total_requests, v1_requests, v2_requests, avg_latency, agreement_rate
# - Log every prediction with PredictionLog

# TODO: Create app_multi = FastAPI(...)
# TODO: Implement /predict with routing
# TODO: Implement /predict-both
# TODO: Implement /metrics

print("TODO: Multi-model API needs implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Run 50 requests and analyze traffic distribution
# Hints:
# - Use TestClient(app_multi)
# - Count v1 vs v2 requests
# - Run 5 A/B comparisons with /predict-both
# - Print traffic split and agreement rate

print("TODO: Traffic simulation needs implementation")
""");


In [ ]:
%%jshell

runPy("""
# TODO: Generate performance comparison report
# Hints:
# - Compare v1 vs v2 latencies
# - Show traffic distribution
# - Create visualization (latency histogram, traffic bar chart)
# - Print summary table

print("TODO: Performance report needs implementation")
""");


---
# Analysis Questions

### Question 1: API Design
Why is input validation important in ML APIs? What specific validation did you implement?

**Your Answer:**
```
TODO: Write your answer here
```

### Question 2: Testing Strategy
What edge cases did you test? Why are they critical for production?

**Your Answer:**
```
TODO: Write your answer here
```

### Question 3: Data Drift
What accuracy degradation did you observe under drift? How does this relate to real-world ADAS challenges?

**Your Answer:**
```
TODO: Write your answer here
```

### Question 4: Monitoring
What metrics would you monitor in production? How would you set alert thresholds?

**Your Answer:**
```
TODO: Write your answer here
```

### Question 5: Deployment
When should you stop a canary deployment? When should you roll back?

**Your Answer:**
```
TODO: Write your answer here
```

---
## Final Summary

In [ ]:
%%jshell

runPy("""
print("""
LAB 2 COMPLETION CHECKLIST
========================

PHASE A: Interface Module
  [ ] CNNModel class implemented
  [ ] Transform pipeline defined
  [ ] /predict, /health, /info endpoints working
  [ ] 8+ test cases passing

PHASE B: Data Monitoring Module
  [ ] BrightnessShiftedDataset implemented
  [ ] ContrastShiftedDataset implemented
  [ ] DriftDetector class with check_accuracy_drift, check_distribution_drift
  [ ] Accuracy evaluated on original + shifted datasets
  [ ] Drift report generated

PHASE C: Deployment Module
  [ ] BaselineModel + DropoutModel created
  [ ] Canary routing implemented (70/30 split)
  [ ] /predict-both for A/B comparison
  [ ] Traffic simulation (50+ requests)
  [ ] Performance comparison report

ANALYSIS
  [ ] All 5 questions answered
  [ ] Visualizations created

DELIVERABLES:
  1. CNN API
  2. API test suite
  3. Drift detection script
  4. Model comparison report

TOTAL POINTS: ___ / 40
""")
""");
